# 报告口径与资源部署

演示独立的缺失与灰样本策略、实际订单指标、加权性能对照、冻结分箱、流式汇总及可迁移部署包。全部使用合成数据；产物写入临时目录。

In [1]:
from pathlib import Path
from tempfile import mkdtemp
import sys, json
root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "hscredit").is_dir())
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
from hscredit import RandomForest, ModelReport, feature_bin_stats, BinStatsAccumulator
from hscredit.database.stream import QueryStream
output = Path(mkdtemp(prefix="hscredit_report_resources_"))
rng = np.random.RandomState(42)
data = pd.DataFrame(rng.normal(size=(360, 3)), columns=["特征一", "特征二", "特征三"])
data["标签"] = (data["特征一"] + .5 * data["特征二"] + rng.normal(size=360) > 0).astype(float)
data["金额"] = rng.randint(100, 1000, 360).astype(float)
data.loc[5, "标签"] = np.nan
data.loc[7, "标签"] = 2  # 业务灰样本，与缺失标签分开。
train, test = data.iloc[:240].copy(), data.iloc[240:].copy()
features = ["特征一", "特征二", "特征三"]
valid = train["标签"].isin([0, 1])
model = RandomForest(n_estimators=10, max_depth=3, random_state=42, n_jobs=1)
model.fit(train.loc[valid, features], train.loc[valid, "标签"], sample_weight=np.where(train.loc[valid, "标签"] == 1, 4., 1.))


RandomForest(history_policy='summary', max_depth=3, max_history=20,
             n_estimators=10, n_jobs=1, random_state=42, target=None,
             weight_type='cost')

## 标签和实际订单口径

默认剔除缺失，灰样本由 `del_grey` 控制。报告主列不会继承训练权重；需要查看加权性能时显式开启对照。

In [2]:
report = ModelReport(
    model, datasets={"训练集": train, "验证集": test}, target="标签", del_grey=True,
    handle_missing_target="drop", rules={"score": [.2, .5, .8], "特征一": [-.5, .5]},
    binning_reference="train", sample_weight={"训练集": np.where(train["标签"] == 1, 4., 1.),
                                              "验证集": np.where(test["标签"] == 1, 4., 1.)},
    show_weighted_metrics=True, n_jobs=1,
)
metrics = report.get_metrics().set_index("统计项")
assert metrics.loc["样本数", "训练集"] == valid.sum()
assert np.isclose(metrics.loc["坏样本率", "训练集"], train.loc[valid, "标签"].mean())
display(report.summary())
display(report.get_bin_table("验证集"))


统计指标     KS                           AUC                         样本数            坏样本率       
数据集     训练集 训练集（加权）    验证集 验证集（加权）    训练集 训练集（加权）    验证集 验证集（加权）  训练集      验证集    训练集    验证集
逾期指标                                                                                        
标签   0.6807  0.6807 0.5335  0.5335 0.9099  0.9099 0.7920  0.7920  238 120.0000 0.5000 0.5333

,分箱标签,样本总数,好样本数,坏样本数,样本占比,好样本占比,坏样本占比,坏样本率,LIFT值,累积LIFT值,坏账改善,累积坏账改善,分档KS值
0,"[0.2, 0.5)",23,20,3,0.1917,0.3571,0.0469,0.1304,0.2446,1.0000,-0.1791,1.0000,0.0000
1,"[0.5, 0.8)",38,22,16,0.3167,0.3929,0.2500,0.4211,0.7895,1.1791,-0.0976,0.7554,0.3103
2,"[0.8, +inf)",59,14,45,0.4917,0.2500,0.7031,0.7627,1.4301,1.4301,0.4160,0.4160,0.4531
3,合计,120,56,64,1.0000,1.0000,1.0000,0.5333,1.0000,1.0000,0.0000,1.0000,0.4531


## JSON 与 Excel 交付

JSON 包含计算结果，Excel 保持原有模型报告章节，配置项注明实际预测函数、缺失处理和分箱基准。

In [3]:
payload = report.to_dict()
(output / "report.json").write_text(json.dumps(payload, ensure_ascii=False, allow_nan=False, indent=2), encoding="utf-8")
report.to_excel(output / "report.xlsx", with_plots=False, include_sample_records=False, mode="strict")
print(output)


C:\Users\18306\AppData\Local\Temp\hscredit_report_resources_8crm48nv


## 流式累计与完整性

这里用本地游标替代真实数据库，验证固定分箱的分块与全量统计一致。生产中直接对数据库返回的 `QueryStream(retain=False)` 使用相同接口。

In [4]:
_, binners = feature_bin_stats(train, "特征一", target="标签", del_grey=True, rules=[-.5, .5], return_binner=True, n_jobs=1)
binner = binners["特征一"]
sample = data.loc[data["标签"].isin([0, 1]), ["特征一", "标签"]]
class Cursor:
    columns = ["特征一", "标签"]
    position = 0
    closed = False
    def fetchmany(self, size):
        part = sample.iloc[self.position:self.position + size]
        self.position += len(part)
        return part
    def close(self):
        self.closed = True
cursor = Cursor()
stream = QueryStream(cursor, chunksize=37, retain=False)
result = stream.aggregate(BinStatsAccumulator(), lambda frame: (binner.transform(frame[["特征一"]])["特征一"], frame["标签"]))
expected = BinStatsAccumulator().update(binner.transform(sample[["特征一"]])["特征一"], sample["标签"]).finalize()
pd.testing.assert_frame_equal(result, expected)
assert cursor.closed and result.attrs["数据源"]["完整"]
assert stream.resource_usage()["保留行数"] == 0
display(result)


,分箱,样本总数,好样本数,坏样本数,样本占比,好样本占比,坏样本占比,坏样本率,分档WOE值,分档IV值,指标IV值,LIFT值,坏账改善,风险拒绝比,累积LIFT值,累积坏账改善,累计风险拒绝比,累积好样本数,累积坏样本数,分档KS值
0,0,101,82,19,0.2821,0.4686,0.1038,0.1881,-1.5070,0.5497,1.1375,0.3680,-0.2484,-0.8804,1.0000,1.0000,NaN,175,183,0.0000
1,1,154,76,78,0.4302,0.4343,0.4262,0.5065,-0.0187,0.0002,1.1375,0.9908,-0.0069,-0.0161,1.2484,0.6320,0.8804,93,164,0.3647
2,2,103,17,86,0.2877,0.0971,0.4699,0.8350,1.5764,0.5877,1.1375,1.6334,0.2558,0.8892,1.6334,0.2558,0.8892,17,86,0.3728


## 部署与资源释放

部署验证只保存预测摘要，不写入验证样本明细。报告释放数据后可读已冻结的整体指标和摘要，明细需要重建报告。

In [5]:
manifest = model.save_deployment(output / "deployment", validation_data=test[features])
restored = RandomForest.load_deployment(manifest, metadata_policy="strict")
np.testing.assert_allclose(restored.predict_proba(test[features]), model.predict_proba(test[features]))
before = report.resource_usage()
summary = report.summary()
report.release_data()
pd.testing.assert_frame_equal(report.summary(), summary)
assert report.resource_usage()["保留行数"] == 0
print("释放前", before)
print("释放后", report.resource_usage())
print("部署完成清单", manifest)


释放前 {'缓存策略': 'full', '数据已释放': False, '数据集数': 2, '保留行数': 360, '估算保留字节': 103793, '估算缓存字节': 25765, '估算规则字节': 36464}
释放后 {'缓存策略': 'full', '数据已释放': True, '数据集数': 0, '保留行数': 0, '估算保留字节': 9510, '估算缓存字节': 3226, '估算规则字节': 64}
部署完成清单 C:\Users\18306\AppData\Local\Temp\hscredit_report_resources_8crm48nv\deployment\model.manifest.json
